In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_dublin_core.xml", "data/metadata/climate_dataset_dublin_core.xml"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")
    else:
        print(f"Already available: {local_path}")

print("Bootstrap complete.")

Downloaded: data/metadata/climate_dataset_dublin_core.xml
Bootstrap complete.



# Day 2: Dublin Core Parsing, Step by Step

In this notebook, we work with a **Dublin Core XML metadata record** for the teaching dataset **"2025 Global Climate Data"**.

## Learning goals
By the end of this notebook, students should be able to:
- understand what a Dublin Core metadata record looks like,
- identify key metadata fields such as title, creator, identifier, and description,
- parse XML metadata step by step in Python,
- convert parsed metadata into a clean table,
- connect metadata fields to the FAIR principles, especially **Findable** and **Reusable**.

## Why this matters
Dublin Core is a simple and widely used metadata standard. It is useful for showing the basic idea of metadata before moving to richer standards such as DataCite and schema.org.



## Step 1 — Locate the metadata file

We first define the path to the local XML file that was downloaded by the bootstrap cell.


In [1]:
from pathlib import Path

# Define path to the XML metadata file
xml_path = Path("data/metadata/climate_dataset_dublin_core.xml")

# Show absolute path for verification/debugging
print("Metadata path:", xml_path.resolve())

# Check if file exists before further processing
print("File exists:", xml_path.exists())

Metadata path: /content/data/metadata/climate_dataset_dublin_core.xml
File exists: False



## Step 2 — Read the XML file as plain text

Before parsing, it is useful to quickly inspect the raw XML so students can see what the metadata record looks like.


In [6]:
raw_xml = xml_path.read_text(encoding="utf-8")
print(raw_xml[:1500])

<?xml version="1.0" encoding="UTF-8"?>
<metadata
  xmlns:dc="http://purl.org/dc/elements/1.1/"
  xmlns:dcterms="http://purl.org/dc/terms/">

  <dc:title>2025 Global Climate Data</dc:title>

  <dc:creator>Global Climate Data Team</dc:creator>

  <dc:publisher>Open Research Repository (Teaching)</dc:publisher>

  <dc:identifier>10.1234/gcd-2025</dc:identifier>

  <dcterms:description>
    A small teaching dataset containing fictional annual climate summaries.
    Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.
  </dcterms:description>

  <dc:subject>climate</dc:subject>
  <dc:subject>temperature</dc:subject>
  <dc:subject>rainfall</dc:subject>
  <dc:subject>CO2 emissions</dc:subject>

  <dc:date>2025-12-31</dc:date>
  <dc:format>text/csv</dc:format>

</metadata>





## Step 3 — Import Python libraries

We use:
- `xml.etree.ElementTree` to parse XML,
- `pandas` to create a readable table.


In [9]:
# XML parsing library for reading and navigating structured metadata files
# (e.g., Dublin Core, DataCite XML) but JSON handling for schema.org (JSON-LD) metadata structures
import xml.etree.ElementTree as ET

import pandas as pd


## Step 4 — Parse the XML and inspect the root element

This helps students understand that XML is a tree structure.


In [11]:
# Parse the XML file into a tree structure
tree = ET.parse(xml_path)

# Get the root element of the XML (entry point for navigation)
root = tree.getroot()

# Shows the main XML tag (e.g., <metadata>, <record>)
print("Root tag:", root.tag)

# Counts immediate child elements under the root (not nested levels)
print("Number of direct child elements:", len(list(root)))

Root tag: metadata
Number of direct child elements: 11



## Step 5 — Understand namespaces and local names

XML metadata files often use namespaces. For example, a tag may look like:

- `{namespace}title`

For teaching purposes, we simplify this by extracting only the **local name** such as `title`, `creator`, or `identifier`.


In [13]:
# Extracts the local tag name by removing XML namespace (e.g., {namespace}title → title)
def local_name(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

# Iterate through XML elements and inspect tag names
# Useful for understanding structure and handling namespaced metadata (Dublin Core, DataCite)
for i, el in enumerate(root.iter()):
    print(local_name(el.tag))
    if i >= 12:  # limit output for quick inspection
        break

metadata
title
creator
publisher
identifier
description
subject
subject
subject
subject
date
format



## Step 6 — Create helper functions

These helper functions make the parsing logic easier to understand:

- `first_text_by_local_name(...)` returns the first matching field,
- `texts_by_local_name(...)` returns all matching values.


In [14]:
def first_text_by_local_name(root: ET.Element, target: str):
    for el in root.iter():
        if local_name(el.tag) == target:
            txt = (el.text or "").strip()
            if txt:
                return txt
    return None

def texts_by_local_name(root: ET.Element, target: str):
    values = []
    for el in root.iter():
        if local_name(el.tag) == target:
            txt = (el.text or "").strip()
            if txt:
                values.append(txt)
    return values


## Step 7 — Extract the main Dublin Core fields

Now we collect the most important metadata fields for this teaching example.


In [15]:
dc_meta = {
    "standard": "Dublin Core",
    "title": first_text_by_local_name(root, "title"),
    "creator": texts_by_local_name(root, "creator"),
    "publisher": first_text_by_local_name(root, "publisher"),
    "identifier": first_text_by_local_name(root, "identifier"),
    "description": first_text_by_local_name(root, "description"),
    "subject": texts_by_local_name(root, "subject"),
    "date": first_text_by_local_name(root, "date"),
    "format": first_text_by_local_name(root, "format"),
    "language": first_text_by_local_name(root, "language"),
    "type": first_text_by_local_name(root, "type"),
}

dc_meta

{'standard': 'Dublin Core',
 'title': '2025 Global Climate Data',
 'creator': ['Global Climate Data Team'],
 'publisher': 'Open Research Repository (Teaching)',
 'identifier': '10.1234/gcd-2025',
 'description': 'A small teaching dataset containing fictional annual climate summaries.\n    Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.',
 'subject': ['climate', 'temperature', 'rainfall', 'CO2 emissions'],
 'date': '2025-12-31',
 'format': 'text/csv',
 'language': None,
 'type': None}


## Step 8 — Convert the metadata into a readable table

A table is often clearer than a raw Python dictionary during class presentation.


In [ ]:
table = pd.DataFrame(
    [
        {"field": "title", "value": dc_meta.get("title")},
        {"field": "creator", "value": "; ".join(dc_meta.get("creator") or [])},
        {"field": "publisher", "value": dc_meta.get("publisher")},
        {"field": "identifier", "value": dc_meta.get("identifier")},
        {"field": "description", "value": dc_meta.get("description")},
        {"field": "subject", "value": "; ".join(dc_meta.get("subject") or [])},
        {"field": "date", "value": dc_meta.get("date")},
        {"field": "format", "value": dc_meta.get("format")},
        {"field": "language", "value": dc_meta.get("language")},
        {"field": "type", "value": dc_meta.get("type")},
    ]
)

table

,field,value
0,title,2025 Global Climate Data
1,creator,Global Climate Data Team
2,publisher,Open Research Repository (Teaching)
3,identifier,10.1234/gcd-2025
4,description,A small teaching dataset containing fictional ...
5,subject,climate; temperature; rainfall; CO2 emissions
6,date,2025-12-31
7,format,text/csv
8,language,None
9,type,None



## Step 9 — Interpret the fields from a metadata perspective

Each field supports a different purpose:

- **title** → helps users discover the dataset,
- **creator** → shows authorship and responsibility,
- **identifier** → provides a persistent reference,
- **description** → explains the content,
- **subject** → improves search and classification,
- **format** → gives a clue about technical usability,
- **date** → supports versioning and context.


In [ ]:
interpretation = pd.DataFrame(
    [
        {"field": "title", "role_in_metadata": "Main descriptive label for discovery"},
        {"field": "creator", "role_in_metadata": "Identifies who created the resource"},
        {"field": "identifier", "role_in_metadata": "Persistent reference to the dataset"},
        {"field": "description", "role_in_metadata": "Explains what the dataset contains"},
        {"field": "subject", "role_in_metadata": "Supports search and thematic grouping"},
        {"field": "format", "role_in_metadata": "Indicates technical format and usability"},
        {"field": "date", "role_in_metadata": "Provides temporal context"},
    ]
)

interpretation

,field,role_in_metadata
0,title,Main descriptive label for discovery
1,creator,Identifies who created the resource
2,identifier,Persistent reference to the dataset
3,description,Explains what the dataset contains
4,subject,Supports search and thematic grouping
5,format,Indicates technical format and usability
6,date,Provides temporal context



## Step 10 — Connect the record to FAIR principles

This is not a full FAIR evaluation, but it shows how metadata supports FAIR.

### Findable
- strengthened by: **title**, **subject**, **identifier**

### Accessible
- can be supported when the identifier points to a reachable landing page or repository record

### Interoperable
- partly supported by structured XML and standard metadata fields

### Reusable
- strengthened by: **description**, **creator**, **date**, **format**


In [ ]:
fair_view = pd.DataFrame(
    [
        {"FAIR_principle": "Findable", "Relevant_fields": "title, subject, identifier", "Comment": "The dataset can be discovered and referenced more easily."},
        {"FAIR_principle": "Accessible", "Relevant_fields": "identifier", "Comment": "Accessibility depends on whether the identifier resolves to an accessible record."},
        {"FAIR_principle": "Interoperable", "Relevant_fields": "standardized XML structure, format", "Comment": "Structured metadata improves machine processing."},
        {"FAIR_principle": "Reusable", "Relevant_fields": "creator, description, date, format", "Comment": "Contextual fields help others understand and reuse the data."},
    ]
)

fair_view

,FAIR_principle,Relevant_fields,Comment
0,Findable,"title, subject, identifier",The dataset can be discovered and referenced m...
1,Accessible,identifier,Accessibility depends on whether the identifie...
2,Interoperable,"standardized XML structure, format",Structured metadata improves machine processing.
3,Reusable,"creator, description, date, format",Contextual fields help others understand and r...



## Step 11 — Discuss possible weaknesses

Even when a metadata file is valid, it may still be limited.

Possible weaknesses to discuss in class:
- missing license information,
- limited provenance details,
- too few subject keywords,
- no explicit relation to versions or related datasets,
- weak description quality.

These points are useful to show students that **metadata quality** matters, not only metadata presence.


In [ ]:
weaknesses = [
    "No explicit license field shown in this simple Dublin Core example.",
    "The description may still be too short for full reuse.",
    "Subject terms may need to be richer and more standardized.",
    "No explicit version or provenance field is included.",
]

for i, item in enumerate(weaknesses, start=1):
    print(f"{i}. {item}")

1. No explicit license field shown in this simple Dublin Core example.
2. The description may still be too short for full reuse.
3. Subject terms may need to be richer and more standardized.
4. No explicit version or provenance field is included.



## Final takeaway

This notebook showed a **step-by-step parsing workflow** for a Dublin Core metadata file:

1. locate the file,  
2. inspect the raw XML,  
3. parse the XML tree,  
4. simplify namespace handling,  
5. extract key fields,  
6. convert the result into a readable table,  
7. interpret the metadata in terms of FAIR support.

This is a good foundation before comparing Dublin Core with **DataCite** and **schema.org**.
